# Rename ICSD Files

Upload one or more **CIF files** and/or **XY/CSV files** (powder diffraction patterns) from the ICSD. The following information is automatically extracted from the CIF file:

- the **chemical formula** (`_chemical_formula_structural`, otherwise `_chemical_formula_sum`),
- the **last name of the first author** (`_citation_author_name`), and
- the **space group** (`_space_group_name_H-M_alt`)

From these, a new file name is generated in the format `Formula_LastName_SpaceGroup`, e.g. `KCo2Se2_Huan_I4mmm`.

**Workflow:**
1. Run the cell below (if necessary, run `pip install ipywidgets` first if it is not installed yet).
2. Select the files using the upload button (CIF and/or XY/CSV files can be uploaded together).
3. Click **"Generate preview"**. For each file, a row appears with the suggested new name, which can be edited manually if needed.
   - If an XY/CSV file cannot be clearly assigned to a CIF file (e.g. because several CIFs were uploaded), a dropdown lets you choose which structure it belongs to.
   - If no CIF file was provided, the original file name is suggested and can be freely overwritten.
4. Check/adjust the target folder (default: subfolder `umbenannt` in the current working directory).
5. Click **"Rename & Save"**. The uploaded files are saved under their new names in the target folder (the originals remain unchanged).

In [1]:
# Falls noch nicht installiert:
# %pip install ipywidgets

import re
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

In [2]:
# --- CIF-Parsing ---------------------------------------------------------

INVALID_CHARS = re.compile(r'[\\/:*?"<>|]')


def sanitize(name: str) -> str:
    """Entfernt Zeichen, die in Dateinamen nicht erlaubt sind."""
    return INVALID_CHARS.sub('', name).strip()


def parse_cif(text: str) -> dict:
    """Liest Summenformel, Erstautor (Nachname) und Raumgruppe aus einem ICSD-CIF-Text."""
    info = {"formula": None, "author": None, "spacegroup": None}

    # Summenformel: bevorzugt die strukturelle Formel (z.B. 'K Co2 Se2'),
    # sonst die Summenformel (z.B. 'Co2 K1 Se2', dabei werden einzelne '1' entfernt).
    m = re.search(r"_chemical_formula_structural\s+'([^']+)'", text)
    if m:
        info["formula"] = m.group(1).replace(' ', '')
    else:
        m = re.search(r"_chemical_formula_sum\s+'([^']+)'", text)
        if m:
            raw = m.group(1)
            raw = re.sub(r'(?<=[A-Za-z])1(?!\d)', '', raw)
            info["formula"] = raw.replace(' ', '')

    # Erstautor: erster Eintrag in der _citation_author_name-Schleife.
    m = re.search(r"_citation_author_name[\s\S]*?(?=\n\s*(?:loop_|_\S|#|$))", text)
    block = m.group(0) if m else text
    authors = re.findall(r"'([^']+)'", block)
    if authors:
        surname = authors[0].split(',')[0].strip()
        info["author"] = sanitize(surname)

    # Raumgruppe: Leerzeichen und Schrägstriche entfernen (z.B. 'I 4/m m m' -> 'I4mmm').
    m = re.search(r"_space_group_name_H-M_alt\s+'([^']+)'", text)
    if not m:
        m = re.search(r"_symmetry_space_group_name_H-M\s+'([^']+)'", text)
    if m:
        sg = m.group(1).replace(' ', '').replace('/', '')
        info["spacegroup"] = sg

    return info


def build_target_stem(info: dict) -> str:
    parts = [
        info.get("formula") or "Formel",
        info.get("author") or "Autor",
        info.get("spacegroup") or "Raumgruppe",
    ]
    return sanitize('_'.join(parts))

In [3]:
# --- Widgets / UI ----------------------------------------------------------

upload = widgets.FileUpload(
    description='Dateien auswählen',
    multiple=True,
    accept='.cif,.csv,.xy,.xye,.txt,.dat,.raw'
)

outdir_box = widgets.Text(
    value=str(Path.cwd() / 'umbenannt'),
    description='Zielordner:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='650px')
)

preview_button = widgets.Button(description='Vorschau erzeugen', button_style='primary')
save_button = widgets.Button(description='Umbenennen & Speichern', button_style='success', disabled=True)
output_area = widgets.Output()
rows_box = widgets.VBox()

state = {"files": {}, "row_widgets": []}


def get_uploaded_files():
    """Liest {Dateiname: Bytes} aus dem FileUpload-Widget, kompatibel mit ipywidgets 7 und 8."""
    files = {}
    val = upload.value
    if isinstance(val, dict):  # ipywidgets 7.x
        for name, item in val.items():
            files[name] = bytes(item['content'])
    else:  # ipywidgets 8.x (Tuple von Dicts)
        for item in val:
            files[item['name']] = bytes(item['content'])
    return files


def on_preview_clicked(b):
    output_area.clear_output()
    rows_box.children = []
    state["files"] = get_uploaded_files()
    state["row_widgets"] = []

    if not state["files"]:
        with output_area:
            print("Bitte zuerst Dateien hochladen.")
        save_button.disabled = True
        return

    # CIF-Dateien parsen -> vorgeschlagene Namen
    cif_targets = {}
    for name, content in state["files"].items():
        if name.lower().endswith('.cif'):
            try:
                text = content.decode('utf-8', errors='ignore')
            except Exception:
                text = content.decode('latin-1', errors='ignore')
            info = parse_cif(text)
            cif_targets[name] = build_target_stem(info)

    data_files = [n for n in state["files"] if not n.lower().endswith('.cif')]
    rows = []

    # Zeilen für CIF-Dateien
    for name, stem in cif_targets.items():
        ext = Path(name).suffix
        stem_box = widgets.Text(value=stem, layout=widgets.Layout(width='420px'))
        row = widgets.HBox([
            widgets.Label(value=name, layout=widgets.Layout(width='260px')),
            widgets.Label('→', layout=widgets.Layout(width='20px')),
            stem_box,
            widgets.Label(ext),
        ])
        rows.append(row)
        state["row_widgets"].append({"original": name, "stem_widget": stem_box, "ext": ext})

    # Zeilen für XY/CSV/etc. Dateien
    cif_stems = list(cif_targets.values())
    for name in data_files:
        ext = Path(name).suffix
        if len(cif_stems) == 1:
            default_value = cif_stems[0]
        elif len(cif_stems) > 1:
            default_value = cif_stems[0]
        else:
            default_value = sanitize(Path(name).stem)

        stem_box = widgets.Text(value=default_value, layout=widgets.Layout(width='420px'))

        if len(cif_stems) > 1:
            dropdown = widgets.Dropdown(options=cif_stems, value=cif_stems[0], layout=widgets.Layout(width='150px'))

            def on_dd_change(change, tb=stem_box):
                tb.value = change['new']

            dropdown.observe(on_dd_change, names='value')
            row = widgets.HBox([
                widgets.Label(value=name, layout=widgets.Layout(width='260px')),
                dropdown,
                widgets.Label('→', layout=widgets.Layout(width='20px')),
                stem_box,
                widgets.Label(ext),
            ])
        else:
            row = widgets.HBox([
                widgets.Label(value=name, layout=widgets.Layout(width='260px')),
                widgets.Label('→', layout=widgets.Layout(width='20px')),
                stem_box,
                widgets.Label(ext),
            ])
        rows.append(row)
        state["row_widgets"].append({"original": name, "stem_widget": stem_box, "ext": ext})

    rows_box.children = rows
    save_button.disabled = False


def on_save_clicked(b):
    output_area.clear_output()
    outdir = Path(outdir_box.value)
    outdir.mkdir(parents=True, exist_ok=True)

    used_names = set()
    with output_area:
        for row in state["row_widgets"]:
            stem = sanitize(row["stem_widget"].value.strip()) or "Datei"
            ext = row["ext"]
            candidate = f"{stem}{ext}"
            counter = 2
            while candidate in used_names or (outdir / candidate).exists():
                candidate = f"{stem}_{counter}{ext}"
                counter += 1
            used_names.add(candidate)

            content = state["files"][row["original"]]
            with open(outdir / candidate, 'wb') as f:
                f.write(content)
            print(f"{row['original']}  →  {candidate}")

        print(f"\nFertig! Dateien gespeichert in: {outdir}")


preview_button.on_click(on_preview_clicked)
save_button.on_click(on_save_clicked)

display(widgets.VBox([
    widgets.HTML("<h3>ICSD-Dateien umbenennen</h3>"),
    upload,
    outdir_box,
    preview_button,
    rows_box,
    save_button,
    output_area,
]))